# Creating a RAG Chatbot Application using Knowledge Base

This application is designed to demonstrate the latest generative AI capabilities using Amazon Bedrock Knowledge Base.

**Amazon Bedrock Agent**: An intelligent chatbot feature that automatically selects between GraphRAG and VectorRAG search methods based on the nature of user queries to provide optimal responses.  
                          GraphRAG specializes in analyzing relationships and connection structures between entities, while VectorRAG is suitable for general information retrieval based on semantic similarity.

**Amazon Bedrock Knowledge Base**: Provides the ability to compare responses from standard Vector RAG and Graph RAG approaches side by side.  
                          This allows users to directly observe how the two search methods yield different results for the same question.

This application is developed based on Streamlit to provide an intuitive web interface, and it leverages the powerful capabilities of the latest large language models (LLMs) such as Claude 3.7 Sonnet through AWS's Bedrock service.  
Users can select their desired demo from the sidebar and check AI responses in real-time.

This demo is a practical example of how companies can build knowledge-based AI solutions through Amazon Bedrock and provide more accurate and relevant responses to complex queries.


## 1. Creating the Chatbot Execution File
Implementation of a RAG application composed of Amazon Bedrock Agent and Knowledge Base as a Streamlit UI-based chatbot for testing.

- Prerequisites
    - Claude Sonnet 3.5 v2 model & Claude instant v1 model need to be activated
    - Need to change **AGENT_ID, VECTOR_RAG_KB_ID, GRAPH_RAG_KB_ID** items in the code below to your current configuration

In [ ]:
%%writefile ../../app.py

import streamlit as st
import boto3
import json
import time
from datetime import datetime
from botocore.client import Config

# AWS Configuration Variables
AWS_REGION = "us-west-2"
AGENT_ALIAS_ID = "TSTALIASID"

AGENT_ID = "AR14QVDQII" # modify with your agent id
VECTOR_RAG_KB_ID = "VXVR4W9Y2O" # modify with your Vector KB ID
GRAPH_RAG_KB_ID = "DBXNEKHXD4" # modify with your Graph KB ID

# Page Configuration
st.set_page_config(page_title="Amazon Bedrock Demos", layout="wide")

# Initialize Session State
if "session_id" not in st.session_state:
    st.session_state.session_id = f"session_{int(time.time())}"

if "messages" not in st.session_state:
    st.session_state.messages = []

if "regular_messages" not in st.session_state:
    st.session_state.regular_messages = []

if "graph_messages" not in st.session_state:
    st.session_state.graph_messages = []

# Create Radio Buttons in Sidebar
with st.sidebar:
    st.header("Select Demo")
    selected_demo = st.radio(
        "Choose a demo to use:",
        ["Amazon Bedrock Agent", "Amazon Bedrock Knowledge Base"]
    )
    
    # Display additional information based on selection
    if selected_demo == "Amazon Bedrock Agent":
        st.header("Agent Information")
        st.markdown(f"**AWS Region**: {AWS_REGION}")
        st.markdown(f"**Agent ID**: {AGENT_ID}")
        st.markdown(f"**Session ID**: {st.session_state.session_id}")
        st.markdown(f"**Current Time**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    else:
        st.header("Knowledge Base Information")
        st.markdown(f"**AWS Region**: {AWS_REGION}")
        st.markdown(f"**Knowledge Base ID**: {VECTOR_RAG_KB_ID} (Vector), {GRAPH_RAG_KB_ID} (Graph)")
        st.markdown(f"**Current Time**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    if st.button("Start New Conversation"):
        st.session_state.session_id = f"session_{int(time.time())}"
        st.session_state.messages = []
        st.rerun()

# Amazon Bedrock Agent Function
def run_agent_chatbot():
    # Initialize Bedrock Client
    bedrock_agent_runtime = boto3.client(
        service_name="bedrock-agent-runtime",
        region_name=AWS_REGION
    )
    bedrock_runtime = boto3.client(
        service_name="bedrock-runtime",
        region_name=AWS_REGION
    )

    # Function to determine semantic search type
    def determine_search_type_semantic(query):
        prompt = f"""
        You are an AI that analyzes the characteristics of questions to recommend the appropriate search method.
        
        Here are two search methods:
        1. GraphRAG: Suitable for relational data such as entity relationships, connection structures, network analysis, path exploration, and knowledge graph patterns.
        2. VectorRAG: Suitable for semantic similarity-based searches such as simple information retrieval, keyword-based queries, document content summarization, and similar concept exploration.
        
        Analyze the following question and determine which search method between "GraphRAG" or "VectorRAG" is more appropriate, then respond in the format below:
        
        Search Method: [GraphRAG or VectorRAG] \n
        Reason: [Explain in one sentence why you chose this search method]
        
        Question: "{query}"
        """
        
        try:
            # Use Claude model from Amazon Bedrock
            response = bedrock_runtime.invoke_model(
                modelId="anthropic.claude-instant-v1",
                body=json.dumps({
                    "prompt": f"\n\nHuman: {prompt}\n\nAssistant:",
                    "max_tokens_to_sample": 200,
                    "temperature": 0,
                    "top_p": 0.9,
                })
            )
            
            response_body = json.loads(response["body"].read())
            response_text = response_body["completion"].strip()
            
            # Extract search method and reason from response
            lines = response_text.split('\n')
            search_type = "VectorRAG"  # default value
            reason = "Default search method."
            
            for line in lines:
                if line.startswith("Search Method:"):
                    search_type_text = line.replace("Search Method:", "").strip()
                    if "GraphRAG" in search_type_text:
                        search_type = "GraphRAG"
                    else:
                        search_type = "VectorRAG"
                elif line.startswith("Reason:"):
                    reason = line.replace("Reason:", "").strip()
            
            return search_type, reason
        except Exception as e:
            st.error(f"Error determining search type: {str(e)}")
            return "VectorRAG", "Using default search method due to error."

    # Function to send query to Bedrock Agent
    def query_bedrock_agent(query, search_type, reason):
        try:
            response = bedrock_agent_runtime.invoke_agent(
                agentId=AGENT_ID,
                agentAliasId=AGENT_ALIAS_ID,
                sessionId=st.session_state.session_id,
                inputText=query,
                enableTrace=False
            )
            
            # Process response
            final_response = ""
            for event in response['completion']:
                if 'chunk' in event:
                    try:
                        chunk_data = json.loads(event['chunk']['bytes'].decode('utf-8'))
                        if 'content' in chunk_data:
                            final_response += chunk_data['content']
                    except json.JSONDecodeError:
                        content = event['chunk']['bytes'].decode('utf-8')
                        final_response += content
            
            # Display search type and reason
            final_response += "\n\n---"
            final_response += f"\n\nSearch Method: {search_type}"
            final_response += f"\n\nSelection Reason: {reason}"
            final_response += "\n\n---"
            
            return final_response
        except Exception as e:
            return f"An error occurred: {str(e)}"

    # Streamlit UI Setup
    st.title("Amazon Bedrock Agent")
    st.markdown("An Agentic chatbot utilizing standard Vector RAG and Graph RAG.")

    # Display chat history
    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])

    # Process user input
    if prompt := st.chat_input("Enter your question..."):
        # Display user message
        st.chat_message("user").markdown(prompt)
        st.session_state.messages.append({"role": "user", "content": prompt})
        
        # Determine semantic search type
        with st.spinner("Analyzing search method..."):
            search_type, reason = determine_search_type_semantic(prompt)
        
        # Display loading indicator
        with st.chat_message("assistant"):
            message_placeholder = st.empty()
            message_placeholder.markdown("🤔 Thinking...")
            
            # Send query to Bedrock Agent
            response = query_bedrock_agent(prompt, search_type, reason)
            
            # Display response
            message_placeholder.markdown(response)
        
        # Add response to chat history
        st.session_state.messages.append({"role": "assistant", "content": response})

# Amazon Bedrock Knowledge Base Function
def run_knowledge_base_demo():
    # Initialize Bedrock Client
    bedrock_config = Config(connect_timeout=120, read_timeout=120, retries={'max_attempts': 0})
    bedrock_runtime = boto3.client(
        service_name="bedrock-runtime",
        region_name=AWS_REGION,
        config=bedrock_config
    )
    bedrock_agent_runtime = boto3.client(
        service_name="bedrock-agent-runtime", 
        region_name=AWS_REGION,
        config=bedrock_config
    )

    # Predefined prompts
    predefined_prompts = [
        "How did the increase in operating costs affect Amazon's other financial metrics?",
        "How has Amazon's total net revenue changed over time?",
        "How did Amazon's online retail service revenue fluctuate quarterly?"
    ]

    # Page header
    st.title("Amazon Bedrock Knowledge Base")
    st.markdown("Comparison of responses between standard Vector RAG and Graph RAG approaches")

In [ ]:
%%writefile ../../app.py

import streamlit as st
import boto3
import json
import time
from datetime import datetime
from botocore.client import Config

# AWS Configuration Variables
AWS_REGION = "us-west-2"
AGENT_ALIAS_ID = "TSTALIASID"

AGENT_ID = "AR14QVDQII" # modify with your agent id
VECTOR_RAG_KB_ID = "VXVR4W9Y2O" # modify with your Vector KB ID
GRAPH_RAG_KB_ID = "DBXNEKHXD4" # modify with your Graph KB ID

# Page Configuration
st.set_page_config(page_title="Amazon Bedrock Demos", layout="wide")

# Initialize Session State
if "session_id" not in st.session_state:
    st.session_state.session_id = f"session_{int(time.time())}"

if "messages" not in st.session_state:
    st.session_state.messages = []

if "regular_messages" not in st.session_state:
    st.session_state.regular_messages = []

if "graph_messages" not in st.session_state:
    st.session_state.graph_messages = []

# Create Radio Buttons in Sidebar
with st.sidebar:
    st.header("Select Demo")
    selected_demo = st.radio(
        "Choose a demo to use:",
        ["Amazon Bedrock Agent", "Amazon Bedrock Knowledge Base"]
    )
    
    # Display additional information based on selection
    if selected_demo == "Amazon Bedrock Agent":
        st.header("Agent Information")
        st.markdown(f"**AWS Region**: {AWS_REGION}")
        st.markdown(f"**Agent ID**: {AGENT_ID}")
        st.markdown(f"**Session ID**: {st.session_state.session_id}")
        st.markdown(f"**Current Time**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    else:
        st.header("Knowledge Base Information")
        st.markdown(f"**AWS Region**: {AWS_REGION}")
        st.markdown(f"**Knowledge Base ID**: {VECTOR_RAG_KB_ID} (Vector), {GRAPH_RAG_KB_ID} (Graph)")
        st.markdown(f"**Current Time**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    if st.button("Start New Conversation"):
        st.session_state.session_id = f"session_{int(time.time())}"
        st.session_state.messages = []
        st.rerun()

# Amazon Bedrock Agent Function
def run_agent_chatbot():
    # Initialize Bedrock Client
    bedrock_agent_runtime = boto3.client(
        service_name="bedrock-agent-runtime",
        region_name=AWS_REGION
    )
    bedrock_runtime = boto3.client(
        service_name="bedrock-runtime",
        region_name=AWS_REGION
    )

    # Function to determine semantic search type
    def determine_search_type_semantic(query):
        prompt = f"""
        You are an AI that analyzes the characteristics of questions to recommend the appropriate search method.
        
        Here are two search methods:
        1. GraphRAG: Suitable for relational data such as entity relationships, connection structures, network analysis, path exploration, and knowledge graph patterns.
        2. VectorRAG: Suitable for semantic similarity-based searches such as simple information retrieval, keyword-based queries, document content summarization, and similar concept exploration.
        
        Analyze the following question and determine which search method between "GraphRAG" or "VectorRAG" is more appropriate, then respond in the format below:
        
        Search Method: [GraphRAG or VectorRAG] \n
        Reason: [Explain in one sentence why you chose this search method]
        
        Question: "{query}"
        """
        
        try:
            # Use Claude model from Amazon Bedrock
            response = bedrock_runtime.invoke_model(
                modelId="anthropic.claude-instant-v1",
                body=json.dumps({
                    "prompt": f"\n\nHuman: {prompt}\n\nAssistant:",
                    "max_tokens_to_sample": 200,
                    "temperature": 0,
                    "top_p": 0.9,
                })
            )
            
            response_body = json.loads(response["body"].read())
            response_text = response_body["completion"].strip()
            
            # Extract search method and reason from response
            lines = response_text.split('\n')
            search_type = "VectorRAG"  # default value
            reason = "Default search method."
            
            for line in lines:
                if line.startswith("Search Method:"):
                    search_type_text = line.replace("Search Method:", "").strip()
                    if "GraphRAG" in search_type_text:
                        search_type = "GraphRAG"
                    else:
                        search_type = "VectorRAG"
                elif line.startswith("Reason:"):
                    reason = line.replace("Reason:", "").strip()
            
            return search_type, reason
        except Exception as e:
            st.error(f"Error determining search type: {str(e)}")
            return "VectorRAG", "Using default search method due to error."

    # Function to send query to Bedrock Agent
    def query_bedrock_agent(query, search_type, reason):
        try:
            response = bedrock_agent_runtime.invoke_agent(
                agentId=AGENT_ID,
                agentAliasId=AGENT_ALIAS_ID,
                sessionId=st.session_state.session_id,
                inputText=query,
                enableTrace=False
            )
            
            # Process response
            final_response = ""
            for event in response['completion']:
                if 'chunk' in event:
                    try:
                        chunk_data = json.loads(event['chunk']['bytes'].decode('utf-8'))
                        if 'content' in chunk_data:
                            final_response += chunk_data['content']
                    except json.JSONDecodeError:
                        content = event['chunk']['bytes'].decode('utf-8')
                        final_response += content
            
            # Display search type and reason
            final_response += "\n\n---"
            final_response += f"\n\nSearch Method: {search_type}"
            final_response += f"\n\nSelection Reason: {reason}"
            final_response += "\n\n---"
            
            return final_response
        except Exception as e:
            return f"An error occurred: {str(e)}"

    # Streamlit UI Setup
    st.title("Amazon Bedrock Agent")
    st.markdown("An Agentic chatbot utilizing standard Vector RAG and Graph RAG.")

    # Display chat history
    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])

    # Process user input
    if prompt := st.chat_input("Enter your question..."):
        # Display user message
        st.chat_message("user").markdown(prompt)
        st.session_state.messages.append({"role": "user", "content": prompt})
        
        # Determine semantic search type
        with st.spinner("Analyzing search method..."):
            search_type, reason = determine_search_type_semantic(prompt)
        
        # Display loading indicator
        with st.chat_message("assistant"):
            message_placeholder = st.empty()
            message_placeholder.markdown("🤔 Thinking...")
            
            # Send query to Bedrock Agent
            response = query_bedrock_agent(prompt, search_type, reason)
            
            # Display response
            message_placeholder.markdown(response)
        
        # Add response to chat history
        st.session_state.messages.append({"role": "assistant", "content": response})

# Amazon Bedrock Knowledge Base Function
def run_knowledge_base_demo():
    # Initialize Bedrock Client
    bedrock_config = Config(connect_timeout=120, read_timeout=120, retries={'max_attempts': 0})
    bedrock_runtime = boto3.client(
        service_name="bedrock-runtime",
        region_name=AWS_REGION,
        config=bedrock_config
    )
    bedrock_agent_runtime = boto3.client(
        service_name="bedrock-agent-runtime", 
        region_name=AWS_REGION,
        config=bedrock_config
    )

    # Predefined prompts
    predefined_prompts = [
        "How did the increase in operating costs affect Amazon's other financial metrics?",
        "How has Amazon's total net revenue changed over time?",
        "How did Amazon's online retail service revenue fluctuate quarterly?"
    ]

    # Page header
    st.title("Amazon Bedrock Knowledge Base")
    st.markdown("Comparison of responses between standard Vector RAG and Graph RAG approaches")

    # Function to retrieve context from Knowledge Base
    def retrieve_from_knowledge_base(query, kb_id, number_of_results=3):
        try:
            # Determine search type based on KB ID
            search_type = "SEMANTIC" if kb_id == GRAPH_RAG_KB_ID else "HYBRID"
            
            # Retrieve context from Knowledge Base
            response = bedrock_agent_runtime.retrieve(
                retrievalQuery={
                    'text': query
                },
                knowledgeBaseId=kb_id,
                retrievalConfiguration={
                    'vectorSearchConfiguration': {
                        'numberOfResults': number_of_results,
                        'overrideSearchType': search_type
                    }
                }
            )
            
            # Extract context from results
            contexts = []
            if 'retrievalResults' in response:
                for result in response['retrievalResults']:
                    if 'content' in result and 'text' in result['content']:
                        contexts.append(result['content']['text'])
            
            return contexts
        
        except Exception as e:
            return f"Error retrieving from Knowledge Base: {str(e)}"

    # Function to query Knowledge Base
    def query_knowledge_base(query, kb_id):
        try:
            # Retrieve relevant context from Knowledge Base
            contexts = retrieve_from_knowledge_base(query, kb_id)
            
            if isinstance(contexts, str) and contexts.startswith("Error"):
                return contexts  # Return error message
            
            # Format prompt with retrieved context
            prompt = f"""
Human: You are an advisor AI system, and provides answers to questions by using fact based when possible.
You're a helpful assistant who loves to respond in Korean.
Use the following pieces of information to provide a detailed answer to the question enclosed in <question> tags.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

<context>
{contexts}
</context>

<question>
{query}
</question>

The response should be specific and use statistics or numbers when possible.

A:"""

            # Use Claude 3.7 Sonnet
            model_id = "us.anthropic.claude-3-7-sonnet-20250219-v1:0" 
            
            response = bedrock_runtime.invoke_model(
                body=json.dumps({
                    "anthropic_version": "bedrock-2023-05-31",
                    "max_tokens": 8192,
                    "messages": [{"role": "user", "content": [{"type": "text", "text": prompt}]}],
                    "temperature": 0.0,
                    "top_p": 0
                }),
                modelId=model_id,
                accept="application/json",
                contentType="application/json"
            )
            
            # Extract response
            response_body = json.loads(response.get('body').read())
            response_text = response_body.get('content')[0]['text']
            
            return response_text
        
        except Exception as e:
            return f"Error querying Knowledge Base: {str(e)}"

    # Create two columns for responses
    col1, col2 = st.columns(2)

    # Display chat history
    with col1:
        st.subheader("Vector RAG")
        for message in st.session_state.regular_messages:
            if message["role"] == "user":
                st.chat_message("user").write(message["content"])
            else:
                st.chat_message("assistant").write(message["content"])

    with col2:
        st.subheader("Graph RAG")
        for message in st.session_state.graph_messages:
            if message["role"] == "user":
                st.chat_message("user").write(message["content"])
            else:
                st.chat_message("assistant").write(message["content"])

    # User selection from predefined prompts
    selected_prompt_index = st.selectbox("Select a question:", options=range(len(predefined_prompts)), format_func=lambda i: predefined_prompts[i])
    
    # Submit button for selected prompt
    if st.button("Submit Question"):
        # Get selected prompt
        user_query = predefined_prompts[selected_prompt_index]
        
        # Clear previous messages
        st.session_state.regular_messages = []
        st.session_state.graph_messages = []
        
        # Add user message to both chat histories
        st.session_state.regular_messages.append({"role": "user", "content": user_query})
        st.session_state.graph_messages.append({"role": "user", "content": user_query})
        
        # Display user message in both columns
        with col1:
            st.chat_message("user").write(user_query)
        with col2:
            st.chat_message("user").write(user_query)
        
        # Query both knowledge bases
        regular_response = query_knowledge_base(user_query, VECTOR_RAG_KB_ID)
        graph_response = query_knowledge_base(user_query, GRAPH_RAG_KB_ID)
        
        # Add assistant responses to chat histories
        st.session_state.regular_messages.append({"role": "assistant", "content": regular_response})
        st.session_state.graph_messages.append({"role": "assistant", "content": graph_response})
        
        # Display assistant responses
        with col1:
            with st.chat_message("assistant"):
                st.write(regular_response)
        
        with col2:
            with st.chat_message("assistant"):
                st.write(graph_response)
        
        # Rerun for UI update
        st.rerun()

# Run selected demo based on radio button selection
if selected_demo == "Amazon Bedrock Agent":
    run_agent_chatbot()
else:
    run_knowledge_base_demo()

## 2. Installing requirements and running the chatbot 
- Run the code below and copy the URL of the web browser where the current SageMaker notebook is open, then modify the last part of the URL as follows to run it: 

```python https://rag-test.studio.us-west-2.sagemaker.aws/jupyterlab/default/[additional]proxy/8080/ ``` 

In [ ]:
# Install packages 
!pip install -r ../requirements.txt 